# 实验二：图像读取、颜色转换、噪声与滤波

参考 [CV_exp / 2nd.exp](https://github.com/Guyudong1/CV_exp/tree/main/2nd.exp)，完成图像读取、颜色转换、加噪、滤波比较和手动中值滤波。原图为 `assets/input.jpg`，尺寸 1080×1578，处理时使用完整分辨率。

处理函数在 `experiment.py` 中，结果图、`metrics.csv` 和 `summary.json` 保存到 `result`。

在仓库根目录执行 `py -3.12 -m venv .venv`，再执行 `.venv\Scripts\python.exe -m pip install -r exp2/requirements.txt` 安装依赖。Notebook 内核选用 `.venv` 中的 Python，从仓库根目录或 `exp2` 目录启动，按顺序运行全部单元格。每次运行会重新写入结果文件。

In [ ]:
from pathlib import Path
import csv
import inspect
import sys

import cv2
from IPython.display import Image, Markdown, display

root = Path.cwd()
exp_dir = root / 'exp2' if (root / 'exp2' / 'experiment.py').exists() else root
assert (exp_dir / 'experiment.py').is_file(), '请从仓库根目录或 exp2 目录启动。'
if str(exp_dir) not in sys.path:
    sys.path.insert(0, str(exp_dir))
from experiment import read_image, run_experiment, manual_median_filter_color

input_path = exp_dir / 'assets' / 'input.jpg'
output_dir = exp_dir / 'result'
SEED = 2026
KERNEL_SIZE = 5
assert input_path.is_file(), f'原图不存在：{input_path}'
print('输入：', input_path)
print('结果目录：', output_dir)

## 1. 读取图像并观察像素

`read_image` 用 OpenCV 解码，得到 `uint8` 的 BGR 三通道数组，也支持中文路径。像素 `[100, 100]` 的行、列索引都从 0 开始。

下方运行整套实验；后续单元格读取结果图和指标。基准随机种子为 2026。

In [ ]:
bgr = read_image(input_path)
summary = run_experiment(input_path, output_dir, seed=SEED, kernel_size=KERNEL_SIZE)
info = summary['input']
print(f"图像：宽 {info['width']} × 高 {info['height']}，{info['channels']} 通道，{info['dtype']}")
print('OpenCV 数组形状 (H, W, C)：', bgr.shape)
print('[100, 100] BGR：', info['pixel_bgr'])
print('[100, 100] RGB：', info['pixel_rgb'])
print('[100, 100] 灰度：', info['pixel_gray'])
display(Image(filename=str(input_path), width=450))

## 2. BGR、RGB 与灰度转换

Matplotlib 按 RGB 显示三通道数组，直接传入 BGR 会使红、蓝颜色互换。转换代码如下：

```python
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
```

灰度按通道权重计算。对比图从左到右为直接显示 BGR、转换后的 RGB 和灰度图。

In [ ]:
print('BGR → RGB → BGR 无损还原：', summary['validation']['bgr_rgb_roundtrip'])
display(Image(filename=str(output_dir / 'color_spaces.png'), width=1100))

## 3. 添加椒盐噪声与高斯噪声

`skimage.util.random_noise` 将 `uint8` 输入转为 `[0, 1]` 浮点数。沿用参考实验的参数：椒盐噪声 `amount=0.4`，盐、椒各半；高斯噪声 `mean=0.2`、`var=0.03`。两类噪声的随机数种子分别为 2026 和 2027。

```python
sp_float = random_noise(rgb, mode='s&p', amount=0.4,
                        rng=np.random.default_rng(SEED))
gaussian_float = random_noise(rgb, mode='gaussian', mean=0.2, var=0.03,
                              rng=np.random.default_rng(SEED + 1))
```

加噪结果乘以 255 并四舍五入转回 `uint8`。椒盐噪声按通道采样，所以受影响的像素比例不等于 40%；原值为 0 或 255 时，采样后也可能保持不变。高斯噪声超出 `[0, 1]` 的值会被裁剪。

In [ ]:
params = summary['parameters']
observations = summary['noise_observations']
print('椒盐 amount：', params['salt_pepper_amount'])
print('高斯 mean / variance：', params['gaussian_mean'], '/', params['gaussian_variance'])
print(f"椒盐实际改变的通道值比例：{observations['salt_pepper_changed_channel_fraction']:.2%}")
print(f"椒盐实际改变的像素比例：{observations['salt_pepper_changed_pixel_fraction']:.2%}")
print(f"高斯上限裁剪比例：{observations['gaussian_upper_clipped_channel_fraction']:.2%}")
print(f"高斯下限裁剪比例：{observations['gaussian_lower_clipped_channel_fraction']:.2%}")
display(Image(filename=str(output_dir / 'noise_comparison.png'), width=1100))

## 4. 均值、中值和高斯滤波对比

三种方法使用相同的加噪图和 5×5 窗口，边界复制边缘像素。高斯滤波设置 `sigmaX=0`，标准差由 OpenCV 根据窗口大小确定。

```python
mean_result = cv2.blur(noisy, (5, 5), borderType=cv2.BORDER_REPLICATE)
median_result = cv2.medianBlur(noisy, 5)
gaussian_result = cv2.GaussianBlur(noisy, (5, 5), 0,
                                   borderType=cv2.BORDER_REPLICATE)
```

对比图上行为椒盐噪声，下行为高斯噪声；每行依次为均值、中值、高斯滤波。指标以原始 RGB 为参考：MSE 越小越好，PSNR、SSIM 越大越好。平均误差是输出减原图的通道值平均，正值表示整体偏亮。

In [ ]:
display(Image(filename=str(output_dir / 'filter_results_2x3.png'), width=1100))
with (output_dir / 'metrics.csv').open(encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
noise_labels = {'salt_pepper': '椒盐', 'gaussian': '高斯'}
method_labels = {'noisy': '加噪原图', 'mean': '均值', 'median': '中值', 'gaussian': '高斯滤波'}
table = ['| 噪声 | 方法 | MSE | PSNR / dB | SSIM | 平均误差 |',
         '| --- | --- | ---: | ---: | ---: | ---: |']
for row in metric_rows:
    table.append(f"| {noise_labels[row['noise']]} | {method_labels[row['method']]} | "
                 f"{float(row['mse']):.2f} | {float(row['psnr_db']):.2f} | "
                 f"{float(row['ssim']):.4f} | {float(row['mean_error']):+.2f} |")
display(Markdown('\n'.join(table)))

## 5. 手动实现彩色中值滤波

对椒盐噪声图的三个通道分别复制边界，再取 5×5 邻域的中值。用 NumPy 构造滑动窗口，每次处理 64 行以减少内存占用，不调用现成滤波函数。

下方列出完整源码，并与相同参数的 OpenCV 中值滤波逐值比较。最大绝对差为 0 表示结果完全相同。

In [ ]:
display(Markdown('```python\n' + inspect.getsource(manual_median_filter_color) + '\n```'))
print('手动中值与 OpenCV 完全一致：', summary['validation']['manual_matches_opencv'])
print('最大绝对差：', summary['validation']['manual_max_abs_difference'])
print('不同的通道值数量：', summary['validation']['manual_different_channel_values'])
display(Image(filename=str(output_dir / 'manual_median.png'), width=1100))

## 结果讨论

下面列出本次运行中 PSNR、SSIM 最高的方法及其相对未滤波图的变化。查看结果图时，还需比较发丝、眼睛边缘和花朵细节是否被平滑。

中值滤波通过邻域排序排除椒盐噪声中的极端值。均值和高斯滤波取邻域平均，容易把脉冲干扰扩散到附近像素；三种方法都可能损失细节。

高斯噪声的 `mean=0.2` 在裁剪前相当于平均增加 51 个灰度级。原图亮部较多，饱和裁剪会丢失信息；空间滤波减弱随机波动后，整体偏亮仍会保留。这里的排序只适用于本图和这组参数，零均值高斯噪声可能有不同结果。

In [ ]:
records = summary['metrics']
for noise in ('salt_pepper', 'gaussian'):
    group = [r for r in records if r['noise'] == noise]
    baseline = next(r for r in group if r['method'] == 'noisy')
    filtered = [r for r in group if r['method'] != 'noisy']
    best_psnr = max(filtered, key=lambda r: r['psnr_db'])
    best_ssim = max(filtered, key=lambda r: r['ssim'])
    print(f"{noise_labels[noise]}噪声：")
    print(f"  未滤波：PSNR {baseline['psnr_db']:.2f} dB，SSIM {baseline['ssim']:.4f}。")
    print(f"  滤波后最高 PSNR：{method_labels[best_psnr['method']]}，"
          f"{best_psnr['psnr_db']:.2f} dB，相对未滤波变化 "
          f"{best_psnr['psnr_db'] - baseline['psnr_db']:+.2f} dB。")
    print(f"  滤波后最高 SSIM：{method_labels[best_ssim['method']]}，"
          f"{best_ssim['ssim']:.4f}，相对未滤波变化 "
          f"{best_ssim['ssim'] - baseline['ssim']:+.4f}。")
    if noise == 'gaussian':
        print(f"  加噪后的平均误差：{baseline['mean_error']:+.2f}。")
        for record in filtered:
            print(f"  {method_labels[record['method']]}后平均误差：{record['mean_error']:+.2f}。")
print(f"高斯上限裁剪通道值比例：{observations['gaussian_upper_clipped_channel_fraction']:.2%}。")
print('输出尺寸及 uint8 类型保持：', summary['validation']['output_shape_preserved'])
print('依赖版本：', summary['versions'])
print('结果图片、指标和摘要已保存至：', output_dir)